# Privacy, GDPR, and AI Governance Demonstration

This notebook demonstrates GDPR-related considerations for the credit
scoring project, including:

- Identifies all Personally Identifiable Information (PII) fields
- Demonstrates pseudonymization/anonymization
- Maps processing activities to relevant GDPR articles
- References the EU AI Act classification for credit scoring systems
- Proposes concrete governance and oversight controls

This analysis is conducted in an academic context based on the dataset
and code in this repository.

## Import and Load the Data

In [5]:
import pandas as pd
import hashlib

df = pd.read_csv("../data/cleaned_credit_applications.csv")
df.head()

,_id,applicant_info_full_name,applicant_info_email,applicant_info_ssn,applicant_info_ip_address,applicant_info_gender,applicant_info_date_of_birth,applicant_info_zip_code,financials_annual_income,financials_credit_history_months,...,savings_balance_missing,debt_to_income_missing,savings_balance_zero,credit_history_suspicious,dob_missing,email_missing,ssn_missing,email_valid,ssn_duplicate,needs_review
0,app_200,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036.0,73000.0,23.0,...,False,False,False,False,False,False,False,True,False,False
1,app_037,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032.0,78000.0,51.0,...,False,False,False,False,False,False,False,True,False,False
2,app_215,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075.0,61000.0,41.0,...,False,False,False,False,False,False,False,True,False,False
3,app_024,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077.0,103000.0,70.0,...,False,False,True,False,False,False,False,True,False,False
4,app_184,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080.0,57000.0,14.0,...,False,False,False,False,False,False,False,True,False,False


## 1. Identification of PII Fields

Under GDPR Article 4(1), personal data is any information relating to an
identified or identifiable natural person.

The dataset contains direct identifiers and quasi-identifiers that
qualify as Personally Identifiable Information (PII).
Identifies all PII fields (names, emails, SSNs, IPs, DOBs).

In [26]:
pii_fields = [
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "applicant_info.date_of_birth",
    "applicant_info.zip_code"
]


pd.DataFrame({
    "PII Field": pii_fields,
    "PII Type": [
        "Direct Identifier",
        "Direct Identifier",
        "Highly Sensitive Identifier",
        "Online Identifier",
        "Personal Attribute",
        "Location-Based Quasi-Identifier"
    ]
})

,PII Field,PII Type
0,applicant_info.full_name,Direct Identifier
1,applicant_info.email,Direct Identifier
2,applicant_info.ssn,Highly Sensitive Identifier
3,applicant_info.ip_address,Online Identifier
4,applicant_info.date_of_birth,Personal Attribute
5,applicant_info.zip_code,Location-Based Quasi-Identifier


## 2. Mapping PII to GDPR Articles

The identified PII fields trigger multiple GDPR obligations related to
lawful processing, data minimization, storage limitation, and data
subject rights.

Maps findings to GDPR articles (lawful basis, minimization, storage limitation).

In [27]:
gdpr_mapping = pd.DataFrame({
    "PII Field": pii_fields,
    "Relevant GDPR Articles": [
        "Art. 6 (Lawful basis), Art. 5(1)(c) (Minimization)",
        "Art. 6 (Lawful basis), Art. 5(1)(c)",
        "Art. 6 (Lawful basis), Art. 5(1)(c), Art. 32 (Security)",
        "Art. 6 (Legitimate interest), Art. 5(1)(c)",
        "Art. 5(1)(c), Art. 5(1)(e) (Storage limitation)",
        "Art. 5(1)(c) (Minimization)"
    ]
})

gdpr_mapping

,PII Field,Relevant GDPR Articles
0,applicant_info.full_name,"Art. 6 (Lawful basis), Art. 5(1)(c) (Minimizat..."
1,applicant_info.email,"Art. 6 (Lawful basis), Art. 5(1)(c)"
2,applicant_info.ssn,"Art. 6 (Lawful basis), Art. 5(1)(c), Art. 32 (..."
3,applicant_info.ip_address,"Art. 6 (Legitimate interest), Art. 5(1)(c)"
4,applicant_info.date_of_birth,"Art. 5(1)(c), Art. 5(1)(e) (Storage limitation)"
5,applicant_info.zip_code,Art. 5(1)(c) (Minimization)


## 3. Pseudonymization and Anonymization

GDPR Article 5(1)(c) and Article 32 encourage data minimization and
pseudonymization to reduce privacy risks.

Below, a direct identifier is pseudonymized to demonstrate privacy-
preserving processing.

### 3.1 Pseudonymize Email

In the dataset, personal attributes such as email addresses are stored
inside a nested `applicant_info` object. To enable GDPR-compliant
pseudonymization, the email field is first extracted and then replaced
with a cryptographic hash.

This approach supports data minimization and security in line with
GDPR Articles 5(1)(c) and 32.

In [12]:
import hashlib
import pandas as pd

def pseudonymize(email):
    if pd.isna(email):
        return None
    return hashlib.sha256(str(email).encode()).hexdigest()

df["email_pseudonymized"] = df["applicant_info_email"].apply(pseudonymize)

df[["applicant_info_email", "email_pseudonymized"]].head()

,applicant_info_email,email_pseudonymized
0,jerry.smith17@hotmail.com,116648a7761525746032d0ab323ceb01f50d11f7935164...
1,brandon.walker2@yahoo.com,c3522c0b54ef9045c73186bcabb53f8e512360ed17e9cc...
2,scott.moore94@mail.com,b299e7d6a37e183bab209eb8df919652117dd16ed16698...
3,thomas.lee6@protonmail.com,6fbd2478748a29faa143392f28d955133e346d09673963...
4,brian.rodriguez86@aol.com,f24e7cc1450ee9aa26b833e0593b2420fbfd59b8ad2636...


### 3.2 Anonymize SSN

The SSN field represents a high-risk identifier. In an analytical
context, it should be removed or irreversibly anonymized.

In [15]:
df = df.drop(columns=["applicant_info_ssn"])
df.columns

Index(['_id', 'applicant_info_full_name', 'applicant_info_email',
       'applicant_info_ip_address', 'applicant_info_gender',
       'applicant_info_date_of_birth', 'applicant_info_zip_code',
       'financials_annual_income', 'financials_credit_history_months',
       'financials_debt_to_income', 'financials_savings_balance',
       'decision_loan_approved', 'loan_purpose', 'decision_interest_rate',
       'decision_approved_amount', 'age_years', 'spend_shopping', 'spend_rent',
       'spend_alcohol', 'spend_dining', 'spend_healthcare', 'spend_fitness',
       'spend_entertainment', 'spend_insurance', 'spend_travel',
       'spend_transportation', 'spend_utilities', 'spend_groceries',
       'spend_education', 'spend_adult_entertainment', 'spend_gambling',
       'annual_income_missing', 'savings_balance_missing',
       'debt_to_income_missing', 'savings_balance_zero',
       'credit_history_suspicious', 'dob_missing', 'email_missing',
       'ssn_missing', 'email_valid', 'ssn_dupli

## 4. Right to Erasure (GDPR Art. 17)

GDPR grants data subjects the right to have their personal data erased.
The following function simulates a deletion request.

In [16]:
def erase_applicant(df, email):
    return df[df["applicant_info_email"] != email]

df_erased = erase_applicant(df, df["applicant_info_email"].iloc[0])
len(df), len(df_erased)

(500, 499)

## 5. EU AI Act Classification

Under the EU AI Act, AI systems used for creditworthiness assessment
and credit scoring are classified as **high-risk AI systems**.

This classification imposes obligations related to:
- risk management
- data governance
- human oversight
- logging and traceability

This project aligns conceptually with these requirements.

## 6. Governance and Oversight Controls

Based on GDPR and EU AI Act requirements, the following governance
controls are recommended.

### Proposed Controls

**Audit Trails**
- Log access to PII fields
- Record model inputs and outputs

**Human Oversight**
- Human review for high-impact or borderline outcomes
- Escalation mechanisms for contested decisions

**Consent & Lawful Basis**
- Explicit consent or contractual necessity for processing
- Documentation of legitimate interest assessments

**Data Retention Policy**
- Define retention periods per PII category
- Automatic deletion after purpose expiration

**Access Control**
- Restrict access to direct identifiers
- Encrypt sensitive attributes at rest and in transit

## 7. Summary

This notebook demonstrates:
- Identification of all PII fields
- Practical pseudonymization and anonymization
- Mapping to GDPR lawful basis, minimization, storage limitation, and erasure
- Alignment with EU AI Act high-risk classification
- Concrete governance and oversight recommendations

Together, these steps illustrate privacy- and governance-by-design.